XGBoost (Extreme Gradient Boosting), Gradient Boosting algoritmasının geliştirilmiş versiyonudur. Özellikle büyük veri setlerinde ve karmaşık modellerde üstün performans gösterir.

#### Gradient Boosting ile başlıca farkları
- Pruning (Budama): XGBoost, ağaçları post-pruning ile optimize eder.
- Regularization: L1/L2 cezaları sayesinde overfitting riskini azaltır.
- Hız: Paralelleştirme ve hafıza optimizasyonlarını içerir.
- Advanced Gain Hesabı: Gain ve similarity ile split kalitesi ölçülür.
- Sigmoid + Log - Odds: Sınıflandırmada residual hesaplaması log-odds ile yapılır.

### XGBoost ile Sınıflandırma

| ID  | Salary (X1) | Credit Score (X2) | Approval (Y) |
| --- | ----------- | ----------------- | ------------ |
| 1   | 30k         | Low               | 0            |
| 2   | 40k         | Low               | 0            |
| 3   | 50k         | Medium            | 1            |
| 4   | 60k         | High              | 1            |
| 5   | 70k         | Medium            | 1            |
| 6   | 80k         | High              | 1            |
| 7   | 90k         | High              | 1            |

#### Adım 1: Başlangıç Tahmini (Base Model)
Hiçbir model henüz kurulmamışken, veriye bakmadan yapılabilecek en iyi şey herkes için aynı değeri tahmin etmektir. Bu değer de genellikle hedef değişkenin ortalaması olur.

Log - Odds = log(Pozitif Oran / Negatif Oran)
Bizim veri setinde:
- Y=1 Pozitif sınıf: 5
- Y=0 Negatif sınıf: 2

Bu durumda:
Log - Odds = log (5/7 / 2/7) = log(2.5) = 0.916
Yani benim başlangıç değerim, herkese atayacağım tahminim F(0) = 0.916

#### Adım 1.1: Sigmoid ile Olasılık
Elimizde her gözlem için bir değer var. F0 = 0.916
Ama bu log-odds değeri şu anda işimize yaramıyor çünkü biz bir olasılık tahmin etmeye çalışıyoruz 0 ve 1 arasında. Dolayısıyla sigmoid fonksiyonu ile bunu olasılığa yani %0 - %100 arasındaki değere sıkıştırıyoruz.

![[Pasted image 20260523222821.png]]

Yukarıdaki formülasyon uygulandığında bu değer 0.714 çıkıyor. Yani modeldeki her sample kişisine %71 ihtimalle onay çıkacak.

#### Bu Adımın Amacı
- Model henüz hiçbir özellik kullanmadan genel bir tahminde bulunuyor.
- Bu başlangıç, modelin üzerine yeni karar ağaçları inşa edeceği *ilk yapı taşıdır*
- Sigmoid ile olasılığa çevirerek, gerçek hayatta müşteri onay alır mı almaz mı sorusuna cevap üretiriz.

*Özet*: XGBoost sınıflandırma problemlerinde tahminler log-odds formatında başlatılır, sonra her adımda sigmoid ile gerçek olasılık elde edilir. Gradient Boosting'den en büyük farkı budur çünkü GB'de doğrudan residual ile çalışılırken burada log-odds dünyasında işlem yapılır.

##### Önemli Not:
Klasik GB, log-loss fonksiyonu kullanıldığında log-odds ve sigmoid de kullanmak zorunda. XGBoost'un farkı, bu işlemleri çok daha sistematik hale getirmesi, regularization ve hessian gibi ek mekanizmalarla daha güçlü ve dengeli bir optimizasyon kullanılmasıdır.

XGBoost ile Gradient Boosting arasındaki temel fark XGB'nin 2. türevi almasıdır. Gradyan zaten 1. türev anlamına geliyor, 2. türev yani XGB'nin kullandığı hise Hessian. Bu türev de hatanın değişim hızını gösterir.

#### Gradyan ve Hessian Nedir?
*Gradyan (1. Türev)*: Hatanın ne kadar büyük olduğunu gösterir. Hem GB hem de XGB bunu yapıyor.
*Hessian (2. Türev)*: Bu hatanın değişim hızını gösterir. XGBoost'ta bu da kullanılıyor. Yani burada tahminler ne kadar yanlışsa düzeltmeyi de büyük yapıyoruz.

Hessian Formülü = - g / h + lambda

Bu sayede:
- Tahminler çok yanlışsa: büyük düzeltme yapılır,
- Tahminler zaten iyiyse: düzeltme az yapılır,
- Model daha dengeli ve kararlı öğrenir.

Örnek:
Gerçek Y = 1,
model tahmini p = 0.9 
Gradyan = 0.9 - 1 = -0.1
Hessian = 0.9 * 0.1 = 0.09
Yani hata küçük, düzeltme de küçük.
#### Adım 2: İlk Residual (Negatif Gradyan) Hesaplama
Negatif gradyan, modelin ne kadar hata yaptığını gösterir:
Rİ = Yi - pi
Burada:
- Yi = Gerçek etiket (0 veya 1)
- pi = Mevcut tahmin edilen olasılık (sigmoid ile hesaplanan)

Bu formülün amacı şudur:
- Eğer R1 negatif ise: Model çok yüksek tahmin yapmış.
- Eğer R1 pozitif ise: Model yetersiz tahmin yapmış demektir.
##### Hesaplama
İlk adımda tüm değerler 0.714 idi.

| ID  | Y (Gerçek) | p (Tahmin) | Residual (R1) |
| --- | ---------- | ---------- | ------------- |
| 1   | 0          | 0.714      | -0.714        |
| 2   | 0          | 0.714      | -0.714        |
| 3   | 1          | 0.714      | +0.286        |
| 4   | 1          | 0.714      | +0.286        |
| 5   | 1          | 0.714      | +0.286        |
| 6   | 1          | 0.714      | +0.286        |
| 7   | 1          | 0.714      | +0.286        |
Sonuçlara göre:
- ID 1 ve ID 2 için: Gerçek değer 0 iken, model 0.714 tahmin etmiş. Yani model çok iyimser davranmış, hatası büyük ve negatif olmuş.
- ID 3 ve ID 4 için: Gerçek değer 1 iken, model ise sadece 0.714 tahmin etmiş. Yani model yetersiz kalmış, hatası 0.286

Bu residual değerleri bize şunu söylüyor:
- Negatif Residual'lar: Bu veri noktalarında modelin tahmini fazla yüksek. Yeni decision tree burada *azaltıcı* etkide bulunmalı.
- Pozitif Residual'lar: Bu veri noktalarında modelin tahmini yetersiz. Yeni decision tree burada *arttırıcı* etkide bulunmalı.
##### Karar Ağacı Ne Öğrenecek?
Bu residual değerleri bizim "etiketimiz" haline gelir. Yani bir sonraki adımda [[Decision Tree]], artık gerçek sınıfları değil, bu residual (hata) değerlerini tahmin etmeye çalışır.
Örneğin ID 1 ve 2 için model, -0.714 tahmin etmeye çalışacak, ID 3 ve 4 için model, 0.286 tahmin etmeye çalışacak.

Bu şu işe yarar:
- İlk decision tree, hataları öğrenerek mevcut tahminlerin daha iyi hale gelmesini sağlar.
- Residual'ların bu şekilde ayrılması sayesinde tree, hangi gözlemler için düşük hangileri için yüksek tahmin yaptı bilgisini öğrenir.
- Bu süreç, modelin adım adım daha iyi hale gelmesini sağlar.

Not: Henüz decision tree kurulmadı. Bu adımda sadece etiketlerimizi (residual'ları) hazırladık. Bu etiketler modelin bir sonraki adımda neyi öğrenmesi gerektiğini belirtiyor.

#### Adım 3: Split, Similarity ve Gain Hesaplama
Önceki adımda tüm gözlemler için residual hesaplamıştık. Şimdi bu residualları kullanarak hangi split'in en iyi sonucu vereeğini belirlemek istiyoruz.
##### Ne yapıyoruz?
Burada henüz decision tree kurmadık. XGBoost'ta bir karar ağacı oluştururken her seferinde hangi özellikten ve hangi eşikten bölünme yapılırsa en yüksek information gain sağlanır bunu hesaplarız. Bu adımın amacı:
- Farklı splitler için similarity ve gain hesaplamak.
- En yüksek gaini sağlayan spliti belirlemek.
- En iyi split belirlendikten sonra tree'yi inşa etmek.

##### Örnek Split: Salary <= 50k
Bu adımda örnek olarak Salary özelliğine göre split atıyoruz.

- Left node: ID 1, 2, 3 (Salary <= 50K)
- Left node: ID 4, 5, 6, 7 (Salary > 50K)

Not: Bu sadece bir öneri. Normalde tüm potansiyel split'ler (örneğin salary ile credit score için farklı eşikler) tek tek denenir ve hepsinin information gaini hesaplanır. Gaini en yüksek olan split seçilir.

![[Pasted image 20260525130033.png]]
#### Adım 4: Split, Similarity ve Gain Hesaplama
Bir önceki adımda en yüksek gain değerine sahip split olarak Salary <= 50k seçildi. Bu split ile decision tree'nin yapısı belirlendi ancak henüz bu node'ların leaf değerleri yani çıktısı hesaplanmadı.
##### Leaf Değerleri Nedir?
Her leaf değeri, o leaf'e düşen gözlemlerin residual hatalarını minimize edecek şekilde hesaplanır.

![[Pasted image 20260525130311.png]]
![[Pasted image 20260525131236.png]]

Güncellenecek Tahminler:
H1 sonrası güncellenmiş tahminler yer alıyor. Sol leafteki negatif tahminler düzeltme alırken, sağ leaftekiler pozitif bir iyileştirme alıyor çünkü yetersiz kalmıştı.

| ID  | F₀    | Leaf   | F₁    | Yeni p |
| --- | ----- | ------ | ----- | ------ |
| 1   | 0.916 | -0.709 | 0.845 | 0.699  |
| 2   | 0.916 | -0.709 | 0.845 | 0.699  |
| 3   | 0.916 | -0.709 | 0.845 | 0.699  |
| 4   | 0.916 | +0.631 | 0.979 | 0.727  |
| 5   | 0.916 | +0.631 | 0.979 | 0.727  |
| 6   | 0.916 | +0.631 | 0.979 | 0.727  |
| 7   | 0.916 | +0.631 | 0.979 | 0.727  |
#### Neden bu adımı yaptık?
İlk tahminden daha hassas hale geldi. H1 ile beraber yani yeni tahmin, model salary'e göre daha da hassaslaştı. Bu, modelin adım adım öğrenmesini sağlıyor.

#### Adım 5: Yeni Residual (Negatif Gradyan) Hesaplama
Bir önceki adımda ilk decision tree'ye göre tahminler güncellendi. Artık model ilk halinden farklı ve daha iyi hale geldi. Şimdi sırada yeni residual'lar ile yeniden hesaplama yapmak var.
#### Neden Bu Adımı Yapıyoruz?
GB temel felsefesi, her iterasyonda modelin yaptığı hataları gidermeye çalışmaktır. İlk ağaç hataları bir miktar düzeltti, ancak model hala mükemmel değil. Yeni decision tree'lerin öğrenebilmesi için güncellenmiş tahminler üzerinden yeni hatalar hesaplamamız gerekiyor.
Bu residual'lar *bir sonraki decision tree'nin öğrenmesi gereken şeyi gösterir.*

Yukarıda daha önce kurduğumuz ilk decision tree'deki tüm adımlar yeni residuallara göre de devam eder.
* Yeni split önerilir (örneğin yine salary <= 50k spliti)
* Similarity ve gain hesaplanır,
* Yeni leaf değerleri hesaplanır,
* Ana model yine güncellenir.
Bu süreç M kadar iterasyon boyunca devam eder.

#### Neden Bu Tekrarlama?
XGBoost'un başarısı, bu iteratif düzeltme yaklaşımından gelir. Model mükemmel hale gelene kadar resiudalları gerçeğe yaklaştırır. Sonuç olarak:
- İlk başta genel bir model,
- Sonra hataları hedef alan karar ağaçları
- Her adımda iyileştirme
- Overfitting azaltmak için küçük learning rate (alpha) ve regularization katsayısı (lamda).